# Tarea 1 — Vbak_Generar_Batch

Simula el extractor SAP depositando un lote nuevo de órdenes en la landing zone.
En producción real, esta tarea NO existiría — SAP deposita el archivo directamente.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CATALOG = "laboratory_sap_dev"
SCHEMA  = "sap_course"
SAP_DATA_PATH = f"/Volumes/{CATALOG}/bronze/curso_databricks"
LANDING_PATH  = f"{SAP_DATA_PATH}/landing/vbak"
STAGING_PATH  = f"{SAP_DATA_PATH}/staging/vbak"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

# batch_id único por corrida — evita colisiones entre ejecuciones del Job
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Simulación: tomar una muestra aleatoria de vbak_bronze como "órdenes nuevas"
(spark.table(f"{CATALOG}.{SCHEMA}.vbak_bronze")
    .drop("_loaded_at", "_source_file")
    .orderBy(F.rand())
    .limit(50)
    .write.mode("overwrite").option("header", "true")
    .csv(STAGING_PATH))

for f in dbutils.fs.ls(STAGING_PATH):
    if f.name.startswith("part-"):
        dbutils.fs.cp(f.path, f"{LANDING_PATH}/batch_{batch_id}/vbak_{batch_id}.csv")

print(f"OK: lote {batch_id} depositado en {LANDING_PATH}/batch_{batch_id}")